## Further exploration: finding an audio input

the first pass (`00` to `03`) only ever traced a visual path, `ol_sensory` -> `vnc_motor`. this file asks whether audio input exists in the data at all, then builds a path from a hearing neuron toward motor output, by hand this time

along the way, it goes back and checks both ends of the visual path, which turns up a mistake worth knowing about

### Init

In [ ]:
import pandas as pd
from pathlib import Path
import pyarrow.dataset as ds
import pyarrow.compute as pc

root = Path.cwd().parent
# root = Path(__file__).resolve().parent.parent

neurons_file = root / 'data' / 'body-annotations-male-cns-v1.0-minconf-0.5.feather'
connections_file = root / 'data' / 'connectome-weights-male-cns-v1.0-minconf-0.5.feather'

df = pd.read_feather(neurons_file)
traced = df[df['status'] == 'Traced']

# more for building
id_to_superclass = traced.set_index('bodyId')['superclass'].to_dict()

# lazy loading, the connections file is too big to load in full
dataset = ds.dataset(connections_file, format="feather")

#### Does audio input exist in this data?

in the annotations cheatsheet, `cb_sensory` is the superclass for sensory input that doesn't come through the vnc, possibly hearing
- `receptorType`: what kind of sense the neuron detects
- `entryNerve`: which nerve carries the neuron in

In [ ]:
cb_sensory = traced[traced['superclass'] == 'cb_sensory']

print(cb_sensory['receptorType'].value_counts())
print(cb_sensory['entryNerve'].value_counts())

> `receptorType` came back empty for `cb_sensory`, `entryNerve` did come back

In [ ]:
# is receptorType populated anywhere?
print(traced[['superclass', 'receptorType']].value_counts())

> `receptorType` isn't broken, it's just populated for `vnc_sensory`, not `cb_sensory`
>
> the values are `ppk23`, `ppk25`, `IR52b`, genes that play roles in pheromone detection, species recognition, and the regulation of courtship behaviors
>
> conclusion: receptor types may not be useful to look at

#### The antennal nerve

`entryNerve` did come back, and `AN` has a large neuron count. `AN` is assumed to be the antennal nerve (from the naming convention, not checked against documentation), and the antenna carries both hearing and smell, so these could be hearing, smell, or a mix of both

check whether `class` or `type` gives a modality clue, since `receptorType` won't

In [ ]:
an_neurons = traced[traced['entryNerve'] == 'AN']
print(an_neurons['class'].value_counts())

> `class` confirms both are there: 2455 `olfactory` and 744 `mechanosensory`
>
> mechanosensory means systems that can respond to sound, touch, motion, etc

In [ ]:
# finer-grained follow-up
print(an_neurons['type'].value_counts().head(20))

> `ORN_*` types dominate (olfactory receptor neurons), matching the `olfactory` class
>
> a few `JO-*` types also show up, Johnston's organ on the antenna, responsible for hearing and sound detection

#### Johnston's organ (JO)

In [ ]:
# drill down on the JO types
jo_neurons = traced[traced['type'].str.contains('JO', na=False)]
print(jo_neurons['type'].value_counts())
print(jo_neurons.shape)

> there are a lot of JO families, maybe for different purposes -> research

JO is split into 5 subgroups by projection pattern in the brain: JO-A through JO-E

only JO-A and JO-B are actually sound/vibration sensors
- JO-A: high frequency tuned, connects strongly to the escape pathway
- JO-B: low frequency tuned, connects to both escape AND the song-relay pathway (song-relay is fly courtship song processing, closest match to "rhythm")

JO-C, JO-D, JO-E are NOT sound, they sense wind/gravity instead

sources:
- Ishikawa et al. 2020 (frontiersin.org/articles/10.3389/fphys.2019.01552)
- Mamiya lab et al., wiring patterns paper (pmc.ncbi.nlm.nih.gov/articles/PMC7676477)

In [ ]:
jo_auditory = jo_neurons[jo_neurons['type'].str.contains('JO-A|JO-B', na=False, regex=True)]
print(jo_auditory['type'].value_counts())
print(jo_auditory.shape)

> JO-B outnumbers JO-A, 88 versus 50, once you sum each family's subtypes
>
> combined with the finding that JO-B is the one wired toward song-relay, and has more neurons, JO-B is a strong candidate for a rhythm-based input stage

#### Side track: checking both ends of the visual path

in the initial exploration, it was concluded that an image had to go from `ol_sensory` -> `vnc_motor`. there was a gap in that, revealed by the exercise above: the subgroups of `ol_sensory` and `vnc_motor` were never checked, so it isn't known whether the correct responsible neurons were used

checking the start first:

In [ ]:
ol_sensory_neurons = traced[traced['superclass'] == 'ol_sensory']
print(ol_sensory_neurons['type'].value_counts())
print(ol_sensory_neurons.shape)

> by sheer luck, the random neuron chosen for `ol_sensory` was of type `R1-R6`, which is responsible for motion and brightness detection, more appropriate for reacting to a moving visual pattern
>
> `R7`/`R8` (color) and `HBeyelet` (probably the extraretinal eyelet, not used for image-forming vision) sit in the same superclass, so the pick could have gone another way

now the other end, `vnc_motor`:

In [ ]:
vnc_motor_neurons = traced[traced['superclass'] == 'vnc_motor']
print(vnc_motor_neurons['type'].value_counts())
print(vnc_motor_neurons.shape)

print(vnc_motor_neurons['exitNerve'].value_counts())
print(vnc_motor_neurons['somaNeuromere'].value_counts())

> `somaNeuromere` splits `vnc_motor` into thoracic (`T1`/`T2`/`T3`, where the legs attach, 500 of 708) and abdominal (`A1` to `A10`)
>
> `exitNerve` points the same way: `ProLN`/`MesoLN`/`MetaLN` (read as leg nerves) versus `AbN*` (read as abdominal nerves), both from the naming

check the end node of the path traced in `03_scaled_lif.ipynb`:

In [ ]:
print(vnc_motor_neurons[vnc_motor_neurons['bodyId'] == 164190][['type', 'exitNerve', 'somaNeuromere']])

> `164190` is `MNad21`, `exitNerve` `AbN2`, `somaNeuromere` `A1`
>
> `Ab` almost certainly stands for abdominal, and `A1` is the first abdominal segment, so the path traced in `03` ends at an abdominal motor neuron, not a leg one
>
> next time, check both the start and end points for where they actually are

this would help find a leg motor instead (needs `G` and `source_id` from `02_graph.ipynb`, so it's left commented out here):

In [ ]:
# leg_motor_ids = vnc_motor_neurons[vnc_motor_neurons['somaNeuromere'].isin(['T1', 'T2', 'T3'])]['bodyId']
# valid_targets = [nid for nid in leg_motor_ids if nid in G.nodes]
# print(len(valid_targets))

# reachable = nx.descendants(G, source_id)
# reachable_motor = [nid for nid in valid_targets if nid in reachable]
# print(len(reachable_motor))

#### Back to the JO neurons: building a path by hand

returning to the JO neurons, the next step is to build a path just like `ol_sensory` -> `vnc_motor`, except this time by hand

`networkx` can find a shortest path between two IDs in one line, but it's not known which two IDs matter, there is no confirmed "sound goes in here" neuron and "movement comes out here" neuron to connect. thus hopping manually is done

In [ ]:
# prints all columns - to look for any JO-B* neurons
# with pd.option_context('display.max_columns', None):
#     print(jo_auditory.head())

jo_id = jo_auditory['bodyId'].iloc[3] # with the above, it was found to be here
print('bodyId we are interested in:', jo_id)

> lands on `JO-B1_a` (`bodyId` 42619), a fully-resolved `JO-B` type rather than one of the `unclear` rows

#### Hop 1: neighbours of the JO neuron

what neurons are outbound from the chosen `JO-B*` one? same neighbour-finding move as before

In [ ]:
filter_expr = (pc.field("body_pre") == jo_id) | (pc.field("body_post") == jo_id)
jo_neighbours = dataset.to_table(filter=filter_expr).to_pandas()
print(jo_neighbours.shape)
print(jo_neighbours)

> some `body_pre`/`body_post` values are enormous, same pattern as with `11139`

In [ ]:
# same move as with 11139 and ol_sensory,
# translate through id_to_superclass before trusting any of these IDs as meaningful
jo_neighbours['pre_type'] = jo_neighbours['body_pre'].map(id_to_superclass)
jo_neighbours['post_type'] = jo_neighbours['body_post'].map(id_to_superclass)

# dropna=False matters here, want to see how many of those 285 rows map to NaN
print(jo_neighbours['pre_type'].value_counts(dropna=False))
print(jo_neighbours['post_type'].value_counts(dropna=False))

In [ ]:
# the strongest connections are the ones most likely to matter for a "does this respond to rhythmic input" story
top_outgoing = jo_neighbours[jo_neighbours['body_pre'] == jo_id].sort_values('weight', ascending=False)
print(top_outgoing.head(10))

> `JO-B1_a` itself classifies as `cb_sensory`, already central-brain sensory (`AN` carries straight to the brain)
>
> the strongest connections are all from the sensory neuron to central brain processing neurons, and it's quite consistent, unlike `ol_sensory`, which went to different superclasses within its top 10 weights

In [ ]:
# in post_type there are 177 NaN rows, a lot, so see if those rows contribute a lot of weight
weighted_targets = (
    jo_neighbours[jo_neighbours['body_pre'] == jo_id]
    .groupby('post_type', dropna=False)['weight']
    .sum()
    .sort_values(ascending=False)
)
print(weighted_targets)

> the 177 NaN rows sum to only 302 weight, compared to `cb_intrinsic` at 490

In [ ]:
# see how heavy the weights of the NaN values are
nan_targets = jo_neighbours[
    (jo_neighbours['body_pre'] == jo_id) & (jo_neighbours['post_type'].isna())
]
print(nan_targets.sort_values('weight', ascending=False))

> the NaN weights are all tiny (9 down to 1), a lot of weak connections

In [ ]:
# for further testing:
nan_ids = nan_targets['body_post'].unique()
print(df[df['bodyId'].isin(nan_ids)][['bodyId', 'status', 'statusLabel']])

# further confirming it
# print(len(nan_ids), 'unique nan targets')
# print(df['bodyId'].isin(nan_ids).sum(), 'of those exist anywhere in df')

> only 2 came back, so many of the connections supposedly exist but are not annotated

In [ ]:
# 527138 is from above, it's weird because it's traced but still empty
print(traced[traced['bodyId'] == 527138][['bodyId', 'status', 'statusLabel', 'superclass', 'class', 'type']])

> otherwise: `JO-B1_a`'s wiring has a strong, concentrated signal into `cb_intrinsic`, with a long tail of weak connections mostly landing on unannotated fragments, and one lone Traced-but-unclassified neuron sitting in that tail

#### Hop 2: the strongest target

the natural next hop: pick the single strongest target, `body_post` 15233, weight 46

In [ ]:
next_id = 15233
filter_expr = (pc.field("body_pre") == next_id) | (pc.field("body_post") == next_id)
next_neighbours = dataset.to_table(filter=filter_expr).to_pandas()
next_neighbours['pre_type'] = next_neighbours['body_pre'].map(id_to_superclass)
next_neighbours['post_type'] = next_neighbours['body_post'].map(id_to_superclass)
print(next_neighbours.shape)
print(next_neighbours['post_type'].value_counts(dropna=False))

> `(1423, 5)`, a lot more neighbours
>
> there are 5 descending neurons in there, which may be the bridge from brain to vnc

next, filter to outgoing only before asking where its signal goes

In [ ]:
outgoing = next_neighbours[next_neighbours['body_pre'] == next_id]
print(outgoing.shape)
print(outgoing['post_type'].value_counts(dropna=False))

weighted_out = (
    outgoing.groupby('post_type', dropna=False)['weight']
    .sum()
    .sort_values(ascending=False)
)
print(weighted_out)

> `cb_intrinsic` is still the biggest bucket by weight, 1148, central-brain intrinsic neurons are talking mostly to other central-brain intrinsic neurons
>
> `descending_neuron` is real but small, 42 out of 2507 total weight

the bridge toward vnc/motor is a trickle, not a highway, right now. worth knowing precisely before deciding whether this is a dead end or just a narrow doorway:

In [ ]:
descending_out = outgoing[outgoing['post_type'] == 'descending_neuron'].sort_values('weight', ascending=False)
print(descending_out)

> these five-digit IDs are clustered in the same narrow range as 10001, which was `DNp01` (the Giant Fiber)

In [ ]:
# just checking the body_post to see what they are
descending_ids = descending_out['body_post'].tolist()
print(traced[traced['bodyId'].isin(descending_ids)][['bodyId', 'type', 'hemibrainType', 'synonyms']])

> two out of five come back with real names attached:
> - `DNp02`, synonym `Matsuo 2016: P3b`
> - `DNg24`, synonym `Matsuo 2016: AMMC-Db7`, a weaker connection but also real
>
> `DNge031`, `DNg29`, `WED180` have no information or publication yet
>
> the `AMMC` in the `DNg24` synonym points at the antennal mechanosensory and motor center, where Johnston's organ neurons arrive, so both look auditory-adjacent
>
> note: published morphology work (Namiki et al. 2018, eLife) places `DNp02` neurites in both the AMMC and the LC4 visual glomerulus, so it may not be purely auditory

#### Hop 3: DNp02

knowing about `DNp02`, that will be used (`bodyId` 10197)

In [ ]:
dnp02_id = 10197

filter_expr = (pc.field("body_pre") == dnp02_id) | (pc.field("body_post") == dnp02_id)
dnp02_neighbours = dataset.to_table(filter=filter_expr).to_pandas()

dnp02_neighbours['pre_type'] = dnp02_neighbours['body_pre'].map(id_to_superclass)
dnp02_neighbours['post_type'] = dnp02_neighbours['body_post'].map(id_to_superclass)

outgoing = dnp02_neighbours[dnp02_neighbours['body_pre'] == dnp02_id]
print(outgoing.shape)

weighted_out = (
    outgoing.groupby('post_type', dropna=False)['weight']
    .sum()
    .sort_values(ascending=False)
)
print(weighted_out)

> there exists a connection to `vnc_motor`! weight 50
>
> however, total outgoing weight here is roughly 14,679, and `vnc_motor` is 50 of that, compare that to `vnc_intrinsic` at 2510. the tiny direct `vnc_motor` link might be a real but minor shortcut, while the "real" pathway is likely one more hop through `vnc_intrinsic`
>
> the NaN bucket is now 9482, `DNp02` is clearly a major hub (6945 outgoing edges vs 285 total neighbour rows for `JO-B1_a`), so a bigger unannotated tail isn't shocking, but not shocking isn't the same as being checked

In [ ]:
nan_out = outgoing[outgoing['post_type'].isna()]
nan_ids = nan_out['body_post'].unique()
print(len(nan_ids), 'unique nan targets')
print(df[df['bodyId'].isin(nan_ids)]['status'].value_counts())

> 6369 unique NaN targets, and only 6 of them exist in `df` at all (2 each: orphan, glia, unimportant)
>
> NaN weight really is just signal going into the unfinished part of the segmentation

In [ ]:
# identify the actual vnc_motor neurons that were hit directly
motor_targets = outgoing[outgoing['post_type'] == 'vnc_motor'].sort_values('weight', ascending=False)
print(traced[traced['bodyId'].isin(motor_targets['body_post'])][['bodyId', 'type', 'instance', 'somaNeuromere']])

> named leg muscle motor neurons were hit, and one jumps out immediately: `TTMn`, the tergotrochanteral motor neuron, the same effector neuron driven by `DNp01` (the Giant Fiber) to trigger a fly's jump-takeoff escape response, the very first neuron looked at
>
> so `DNp02` also drives that same jump muscle, alongside genuine leg extensor and leg flexor motor neurons
>
> every single one of these sits at `somaNeuromere == T2`, the mesothoracic segment, which controls the middle pair of legs, the fly's primary walking/jumping legs

#### Side track: vnc_intrinsic

the largest weights that `DNp02` connects to are `vnc_intrinsic`, so look at those

In [ ]:
vnc_intrinsic_targets = outgoing[outgoing['post_type'] == 'vnc_intrinsic'].sort_values('weight', ascending=False)
print(vnc_intrinsic_targets.shape)
print(vnc_intrinsic_targets.head(10))

In [ ]:
# don't stop at the superclass label, look at the actual named identities of the strongest few
top_vnc_ids = vnc_intrinsic_targets.head(10)['body_post']
print(traced[traced['bodyId'].isin(top_vnc_ids)][['bodyId', 'type', 'instance', 'somaNeuromere', 'subclass']])

> every single one of these is named `IN___`, this is where the real processing muscle is, not the small direct shortcut
>
> 7 of these 10 land in `T1`, only 3 in `T2`, so this strongest `vnc_intrinsic` bucket looks like it's mostly driving a different pair of legs (but see below, the soma segment turns out not to predict which legs an interneuron drives)

In [ ]:
# IN00A010 shows up twice, once in T1 (801689) and once in T2 (919595)
print(traced[traced['bodyId'].isin([801689, 919595])][['bodyId', 'type', 'serialMotif', 'mancSerial']])

In [ ]:
in23b001_id = 800373  # strongest single connection, weight 120, T1
filter_expr = (pc.field("body_pre") == in23b001_id) | (pc.field("body_post") == in23b001_id)
in_neighbours = dataset.to_table(filter=filter_expr).to_pandas()
in_neighbours['post_type'] = in_neighbours['body_post'].map(id_to_superclass)

out_in = in_neighbours[in_neighbours['body_pre'] == in23b001_id]
print(out_in.groupby('post_type', dropna=False)['weight'].sum().sort_values(ascending=False))

> `IN23B001` (the T1 branch just checked) drives `vnc_motor` directly, weight 225
>
> that's over 4x stronger than the 50-weight direct motor link found from `DNp02`. `DNp02` mostly drives interneurons, and those interneurons are where the strong, reliable motor drive actually comes from, the direct `DNp02` to motor link was the minor shortcut, and this interneuron layer is the main event

In [ ]:
# which specific motor neurons IN23B001 lands on, same move as before
motor_from_in = out_in[out_in['post_type'] == 'vnc_motor']
print(traced[traced['bodyId'].isin(motor_from_in['body_post'])][['bodyId', 'type', 'instance', 'somaNeuromere']])

> worth noting: no T1 targets at all, despite `IN23B001` itself sitting in T1
>
> instead it's split between T2 (13) and T3 (19), the middle and hind legs
>
> this lines up with the `serialMotif` label from before, `vnc convergent`, convergent is the right word for a hub that gathers/distributes across segments rather than staying local

#### The two pathways

> two full, verified, named pathways, both starting from the same auditory input:
> - direct link (weak, weight 50): `JO-B1_a -> 15233 -> DNp02 -> TTMn/extensor/flexor motor neurons`, all in T2 (middle leg)
> - via an interneuron (strong, weight 225): `JO-B1_a -> 15233 -> DNp02 -> IN23B001 -> motor neurons`, `IN23B001` sits in T1 but its motor targets are in T2 and T3 (middle and hind legs)
>
> the second path is the stronger, more biologically convincing one: sound drives an interneuron layer, which drives motor output, the actual expected shape of a reflex circuit, not just a lucky direct wire

#### Building the graph

this gives a complete, hand-verified, four-hop pathway with a real biological shape. the weights below were typed in by hand from the printouts above

In [ ]:
import networkx as nx

G = nx.DiGraph()

confirmed_path_edges = [
    (jo_id, 15233, 46),
    (15233, 10197, 12),        # DNp02
    (10197, 800373, 120),      # IN23B001
]

for pre, post, w in confirmed_path_edges:
    G.add_edge(pre, post, weight=w)

for _, row in motor_from_in.iterrows():
    G.add_edge(in23b001_id, row['body_post'], weight=row['weight'])

print(G.number_of_nodes(), 'nodes,', G.number_of_edges(), 'edges')
print(list(nx.all_simple_paths(G, source=jo_id, target=800589)))  # one example flexor MN

### Conclusion

what this file actually established:
- audio input does exist in the data: the antennal nerve carries both smell and hearing, and within it only the `JO-A`/`JO-B` subtypes (138 neurons) are sound sensors. `JO-B` (88 vs 50) is the better candidate for a rhythm-based input, since it's wired toward the song-relay pathway
- a hand-built, four-hop pathway exists from a fully-resolved `JO-B` neuron (`JO-B1_a`) to leg motor neurons, through `15233` and `DNp02`, either by a weak direct link (weight 50, including `TTMn`, the jump motor neuron the Giant Fiber also drives) or by a much stronger route through the interneuron `IN23B001` (weight 225), landing in the middle and hind legs
- checking both ends of the visual path caught a mistake: the endpoint used in `03` (`164190`) is an abdominal motor neuron, not a leg one, so the timing results there describe a path to the wrong body part, even though the mechanics themselves still hold

what's still open:
- this pathway is structural only, nothing has been simulated through it yet, and its weights were typed in by hand
- `DNp02` may not be purely auditory, published work places it in a visual glomerulus too (Namiki et al. 2018)

so next:
- run the scaled LIF from `03` on this audio path
- redo the visual path with a leg motor target (the commented-out code above)